# Nonlinear regression 
## Introduction


## Mathematical foundation
### Various nonlinear models


### A biological example


### Least squares estimation


## Nonlinear regression in Python
### Exploring the data


In [ ]:
import numpy as np
import pandas as pd

relaxation = np.array(
    [2.6, 10.5, 15.8, 21.1, 36.8, 57.9, 73.7, 89.5, 94.7, 100, 100])
norepi_log = np.arange(-8, -2.5, .5)  # Log scale concentrations

data_doseresponse = pd.DataFrame({
    'norepi_log': norepi_log,
    'relaxation': relaxation})
# Not strictly necessary to prepare a DataFrame
print("Dose-response dataset:")
print(data_doseresponse)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# For plotting, we'll need linear concentrations
norepi_lin = 10**norepi_log

plt.figure(figsize=(3.5, 3))

# Create the plot
plt.plot(norepi_lin, relaxation, 'bo')
plt.title("Relaxation dose-response")
plt.xlabel("[Norepinephrine, M] (log scale)")
plt.xscale('log') 
plt.ylabel(r"% relaxation");

### Fitting the Hill equation
#### Defining our model function


In [ ]:
def hill_equation(x, bottom, top, logEC50, hill_slope):
    """
    This function defines the general Hill equation (4PL).

    Args:
        x: the logarithm of norepinephrine concentration
        bottom: the baseline relaxation (Y0)
        top: the maximum relaxation (Ymax)
        logEC50: the logarithm of the EC50
        hill_slope: the Hill slope (nH)

    Returns:
        The predicted relaxation values.
    """
    return bottom + (top - bottom) / (1 + 10**((logEC50 - x) * hill_slope))

In [ ]:
def hill_equation_bottom_zero(x, top, logEC50, hill_slope):
    """
    This function defines the Hill equation with bottom fixed at 0.

    Args:
        x: the logarithm of norepinephrine concentration
        top: the maximum relaxation (Ymax)
        logEC50: the logarithm of the EC50
        hill_slope: the Hill slope (nH)

    Returns:
        The predicted relaxation values.
    """
    return top / (1 + 10**((logEC50 - x) * hill_slope))  # Bottom fixed to 0

In [ ]:
def hill_equation_3pl(x, bottom, top, logEC50):
    """
    This function defines the 3-parameter logistic (3PL) equation.

    Args:
        x: the logarithm of norepinephrine concentration
        bottom: the baseline relaxation (Y0)
        top: the maximum relaxation (Ymax)
        logEC50: the logarithm of the EC50

    Returns:
        The predicted relaxation values.
    """
    return bottom + (top - bottom) / (1 + 10**(logEC50 - x))
    # Hill slope fixed to 1

#### Providing initial parameter guesses


In [ ]:
# Initial guess for the parameters for top, LogEC50, and hill_slope,
# respectively, to be used in `hill_equation_bottom_zero`(bottom=0)
initial_guesses = [100, -6, 1]
print("Initial guess:")
print(f" Top = {initial_guesses[0]}")
print(f" LogEC50 = {initial_guesses[1]}")
print(f" Hill slope = {initial_guesses[2]}")

#### Fitting the model to the data


In [ ]:
from scipy.optimize import curve_fit

# Fit the 4PL model to the data (bottom is fixed to 0 there)
best_vals, covar = curve_fit(
    hill_equation_bottom_zero,
    norepi_log,
    relaxation,
    p0=initial_guesses)

# Print the best-fit parameter values
print("Best-fit values of parameters (4PL model with fixed bottom):")
print(f" Top = {best_vals[0]:.1f}")
print(f" LogEC50 = {best_vals[1]:.2f}")
print(f" EC50 = {10**best_vals[1]:.2e}")
print(f" Hill slope = {best_vals[2]:.3f}")

#### Extracting standard errors


In [ ]:
# Calculate the standard errors of the parameters
standard_errors = np.sqrt(
    np.diag(covar)  # Extract the diagonal elements of the covariance matrix
)

# Print the standard errors
print("\nStandard errors of parameters:")
print(f" Top: {standard_errors[0]:.2f}")
print(f" LogEC50: {standard_errors[1]:.3f}")
print(f" Hill slope: {standard_errors[2]:.4f}")

## Advanced nonlinear regression with lmfit


### Setting parameter bounds and constraints


In [ ]:
from lmfit import Model

# Create a model object giving the 4PL hill_equation function
model = Model(hill_equation)

# Create parameter objects, setting intial bounds and constraints
params = model.make_params(top=100, logEC50=-6, hill_slope=1)

# Set initial value for bottom to zero and fix it
params.add('bottom', value=0, vary=False)
# params['bottom'].value = 0     # Set initial value for bottom to zero
# params['bottom'].vary = False  # Fix its value

params['top'].min = 0            # Non-negative
params['hill_slope'].min = 0     # Non-negative

# Display the initial parameters set
print(f"Parameters set for the model {model.name}")
params.pretty_print(columns=['init_value', 'min', 'max', 'vary'])

### Fitting the model to data


In [ ]:
# Fit the model to the data
result = model.fit(
    data=relaxation,
    x=norepi_log,
    params=params,)

# Display the optimized parameters
result.params.pretty_print(columns=['value', 'stderr', 'init_value'])

### Accessing richer fit statistics and diagnostics


In [ ]:
print("lmfit fit report:")
print(result.fit_report())

In [ ]:
# Access specific fit statistics
print(f"Best-fit values: ")
print(pd.Series(result.best_values))  # Dictionnary intially returned
print()
print(f"DF={result.nfree:n}")
print(f"Absolute sum of squares (χ²)={result.chisqr:.2f}")
print(f"Reduced chi-squared={result.redchi:.2f}")
print(f"AIC={result.aic:.2f}")
print(f"BIC={result.bic:.2f}")
print(f"R²={result.rsquared:.3f}")
print()
# Access parameter correlation matrix
print("Parameter correlation matrix:")
print(result.covar.round(3))

In [ ]:
# Print the fit results for better readability
#result.result

### Predicting values


In [ ]:
# Predict the y-value for a new x-value using curve_fit results
new_x = -6.0  # Log[Norepinephrine] = -6.0
predicted_y_scipy = hill_equation_bottom_zero(new_x, *best_vals)
# Array unpacking

print("Prediction using SciPy curve_fit results (array unpacking):")
print(f"Predicted y-value for x={new_x}: {predicted_y_scipy:.2f}")

In [ ]:
# Predict the y-value for a new x-value using lmfit best-fit parameters
predicted_y_lmfit = hill_equation(new_x, **result.params)
# Dictionary unpacking

print("Prediction using lmfit best-fit parameters (dictionary unpacking):")
print(f"Predicted y-value for x={new_x}: {predicted_y_lmfit:.2f}")

In [ ]:
# Predict the y-value for a new x-value using lmfit result
predicted_y_lmfit_eval = result.eval(x=new_x)

print("Prediction using lmfit best-fit parameters (eval method):")
print(f"Predicted y-value for x={new_x}: {predicted_y_lmfit_eval:.2f}")

In [ ]:
# Add a new column into the original dataframe
data_doseresponse['y_predicted'] = result.eval()

print("Predicted values using original independent variable:")
print(data_doseresponse[['norepi_log', 'y_predicted']])

## Assessing the goodness of fit
### Residual analysis in nonlinear regression


#### Plotting residuals vs. predicted values


In [ ]:
# Calculate the residuals
residuals = result.residual
#residuals = result.best_fit - relaxation

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(6, 3), sharey=True)

# Manual plot on the left subplot
axes[0].plot(result.best_fit, residuals, 'bo')
axes[0].axhline(y=0, color='orange', linestyle='--')
axes[0].set_xlabel(r"Predicted % relaxation")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residual plot (Matplotlib)")

# lmfit plot on the right subplot
result.plot_residuals(ax=axes[1], title='Residual plot (lmfit)')
axes[1].set_xlabel("Log[Norepinephrine]")

plt.tight_layout();

#### Examining patterns in residuals


### Calculating R-squared


In [ ]:
# Define functions to compute RSS and estimate y
def compute_rss(y_estimate, y):
    return sum(np.power(y - y_estimate, 2))

# Calculate RSS using the compute_rss() function from the previous chapters
rss = compute_rss(result.best_fit, relaxation)

# Calculate TSS
tss = np.sum(np.power(relaxation - np.mean(relaxation), 2))

# Calculate and print R-squared
print(f"TSS (manual) = {tss:.2f}")
print(f"RSS (manual) = {rss:.3f}")
print(f"χ² (lmfit) = {result.chisqr:.3f}")
print()
print(f"R² (manual) = {1 - rss/tss:.4f}")
print(f"R² (lmfit) = {result.rsquared:.4f}")

## Visualizing and interpreting nonlinear models
### Plotting the optimal curve


In [ ]:
#| fig-subcap: 
#|   - "Predicted values using SciPy"
#|   - "Predicted values using lmfit"
#|   - "Optimal curve using plot_fit"
#| layout-ncol: 3

# Generate x-values for the curve in the range of norepi_log (log scale)
x_range = np.linspace(norepi_log.min(), norepi_log.max(), 100)

# Plot 1: Fitted `hill_equation_bottom_zero` (SciPy parameters)
# Calculate predicted values using the unpack (*) best-fit parameters array
predicted_values = hill_equation_bottom_zero(x_range, *best_vals)

plt.figure(figsize=(3.5, 3))
plt.plot(norepi_lin, relaxation, 'ko', label="Data")
plt.plot(10**x_range, predicted_values, 'r-', lw=2, label="Fitted curve")
plt.xscale('log')
plt.xlabel("[Norepinephrine, M]")
plt.ylabel(r"% relaxation")
plt.title("SciPy best-fit parameters")
plt.legend()
plt.show()

# Plot 2: Fitted `hill_equation` (lmfit parameters)
# Calculate predicted values unpacking best-fit parameters dictionay
predicted_values = hill_equation(x_range, **result.params)
# We cannot reuse the predicted y-values from previous code 
# as x-range is different

plt.figure(figsize=(3.5, 3))
plt.plot(norepi_lin, relaxation, 'ko', label="Data")
plt.plot(10**x_range, predicted_values, 'r-', lw=2, label="Fitted curve")
plt.xscale('log')
plt.xlabel("[Norepinephrine, M]")
plt.ylabel(r"% relaxation")
plt.title("lmfit best-fit parameters")
plt.legend()
plt.show()

# Plot 3: Fitted `hill_equation` (`result.plot_fit`)
plt.figure(figsize=(3.5, 3))
result.plot_fit(
    datafmt='ko',
    xlabel="Log[Norepinephrine, M]",
    ylabel=r"% relaxation",
    title="lmfit plot_fit",
    fit_kws={'lw': 2, 'c': 'red'},)
plt.show()

In [ ]:

result.plot(
    datafmt='ko',
    fitfmt='r-',
    xlabel="Log[Norepinephrine, M]",
    ylabel=r"% relaxation",
    fit_kws={'lw': 2},
    fig_kws={'figsize': (3.5, 3.5)});

### Confidence intervals


In [ ]:
np.set_printoptions(suppress=False, precision=4)
from scipy.stats import t as t_dist

# Confidence level
α = 0.05

# Calculate the critical t-value for a 95% confidence interval
t_critical = t_dist.ppf((1 + (1-α)) / 2, result.nfree)

# Function to calculate confidence intervals
def get_ci(estimate, se):
    return np.array((estimate-t_critical*se, estimate+t_critical*se))

# Print the 95% confidence intervals of the parameters
print("95% CI of parameters:")
for i, param_name in enumerate(['Top', 'LogEC50', 'Hill slope']):
    ci = get_ci(best_vals[i], standard_errors[i])
    print(f" {param_name} =", ci.round(2))

    # Special case for EC50 (convert to linear scale)
    if param_name == 'LogEC50':
        print(f" EC50 = {(10**ci)}")

In [ ]:
print("Report for confidence intervals:")
print(result.ci_report())

In [ ]:
print("Confidence intervals (customized report):")
# If sigmas < 1 than it corresponds to %CI, else it corresponds to n × σ
# e.g. sigmas=[0.95] is the same as sigmas=[1.96]
print(result.ci_report(with_offset=False, ndigits=3, sigmas=[0.95]))
#result.conf_interval(p_names=['top'], sigmas=[1.96])

### Visualizing confidence bands


In [ ]:

# Create a figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(6.5, 2.25), sharex=True, sharey=True)

# Define a function to plot the confidence band for a given parameter
def plot_confidence_band(ax, param_name, param_index):
    # Calculate the lower and upper confidence bounds for the curve
    params_low = best_vals.copy()
    params_low[param_index] = (
        best_vals[param_index] - t_critical * standard_errors[param_index])
    lower_bound = hill_equation_bottom_zero(x_range, *params_low)

    params_high = best_vals.copy()
    params_high[param_index] = (
        best_vals[param_index] + t_critical * standard_errors[param_index])
    upper_bound = hill_equation_bottom_zero(x_range, *params_high)

    # Plot the data, fitted curve, and confidence band
    ax.plot(norepi_lin, relaxation, '.', label="Data")
    ax.plot(10**x_range, predicted_values, 'r-')
    ax.fill_between(
        10**x_range, lower_bound, upper_bound,
        color='yellowgreen',)
    ax.set_xscale('log')
    ax.set_xlabel("[Norepinephrine]")
    ax.set_ylabel(r"% relaxation")
    ax.set_title(f"Varying {param_name}", fontsize=11)

# Plot the confidence bands for each parameter
plot_confidence_band(axes[0], 'top', 0)
plot_confidence_band(axes[1], 'logEC50', 1)
plot_confidence_band(axes[2], 'hill_slope', 2)

plt.tight_layout();

In [ ]:

# Calculate the uncertainty in the predicted values
σ=0.95  # Same as σ=1.96
dely = result.eval_uncertainty(sigma=σ)

plt.figure(figsize=(3.5, 3))

# Plot the fitted curve, confidence band, and the data
result.plot_fit(
    numpoints=100,
    fit_kws={'lw': 2})
plt.fill_between(
    norepi_log,
    result.best_fit - dely,
    result.best_fit + dely,
    color='greenyellow',
    label="confidence band")
plt.xlabel('Log[Norepinephrine, M]')
plt.ylabel('Relaxation (%)')
plt.title("95% confidence band")
plt.legend();

### Exploring the parameter space visually


In [ ]:
#| fig-subcap: 
#|   - "Initial guess fit"
#|   - "Lower EC50 fit"
#|   - "3PL fit"
#| layout-ncol: 3

# Plot 1: Original data and initial guess
plt.figure(figsize=(3.5, 3))
result.plot_fit(
    datafmt='ko',
    fitfmt='r-',
    title='Initial guess',
    xlabel="Log[Norepinephrine, M]",
    ylabel=r"% relaxation",
    show_init=True,  # Include the initial guess fit
    fit_kws={'lw': 2},
    init_kws={'c': 'grey'})
plt.show()

# Plot 2: Lower EC50
log_ec50_eval = -6.8  # LogEC50 to evaluate
plt.figure(figsize=(3.5, 3))
result.plot_fit(
    datafmt='ko',
    fitfmt='r-',
    title='Lower EC50',
    xlabel="Log[Norepinephrine, M]",
    ylabel=r"% relaxation",
    show_init=False,  # Don't include the initial fit
    fit_kws={'lw': 2})
plt.plot(
    norepi_log,
    result.eval(logEC50=log_ec50_eval,),
    label=f'LogEC50={log_ec50_eval}')
plt.legend()
plt.show()

# Plot 3: Hill slope = 1
hill_slope_eval=1
plt.figure(figsize=(3.5, 3))
result.plot_fit(
    datafmt='ko',
    fitfmt='r-',
    title='3PL',
    xlabel="Log[Norepinephrine, M]",
    ylabel=r"% relaxation",
    show_init=False,  # Don't include the initial fit
    fit_kws={'lw': 2})
plt.plot(
    norepi_log,
    result.eval(hill_slope=hill_slope_eval),  # 3PL equation
    label=f'Hill slope={hill_slope_eval}')
plt.legend()
plt.show()

## Comparing nonlinear models


### Nested models


### The extra sum-of-squares F-test
#### Fitting the models for comparison


In [ ]:
def hill_equation_3pl_bottom_zero(x, top, logEC50):
    """
    Defines the 3-parameter logistic (3PL) equation with bottom fixed at 0.

    Args:
        x: the logarithm of norepinephrine concentration
        top: the maximum relaxation (Ymax)
        logEC50: the logarithm of the EC50

    Returns:
        The predicted relaxation values
    """
    return top / (1 + 10**(logEC50 - x))
    # hill_slope is fixed to 1, bottom to 0

In [ ]:
# Create a model object for the variable slope reduced 4PL model
model_variable_slope = Model(hill_equation_bottom_zero)

# Create a model object for the new fixed slope reduced 3PL model
model_fixed_slope = Model(hill_equation_3pl_bottom_zero)

# Initial guesses for the parameters
params_variable_slope = model_variable_slope.make_params(
    top=100, logEC50=-5, hill_slope=0.6)
params_fixed_slope = model_fixed_slope.make_params(
    top=100, logEC50=-5)

# Fit the models to the data
result_variable_slope = model_variable_slope.fit(
    data=relaxation, x=norepi_log, params=params_variable_slope)
result_fixed_slope = model_fixed_slope.fit(
    data=relaxation, x=norepi_log, params=params_fixed_slope)

In [ ]:
print("lmfit fit report for the variable Hill slope reduced 4PL model \
(bottom=0):")
print(result_variable_slope.fit_report())

In [ ]:
print("lmfit fit report for the fixed Hill slope reduced 3PL model \
\n(bottom=0, hill_slope=1):")
print(result_fixed_slope.fit_report())

#### Comparing the residuals


In [ ]:

from matplotlib import gridspec

# Create a GridSpec object to define the layout
gs = gridspec.GridSpec(2, 2, width_ratios=[3,2])

# Create the axes using the GridSpec
fig = plt.figure(figsize=(6, 3.5))

# Main plot spans both rows in the first column
ax_main = fig.add_subplot(gs[:, 0])
# Variable slope residuals in the top-right
ax_resid_var = fig.add_subplot(gs[0, 1])
# Fixed slope residuals in the bottom-right
ax_resid_fix = fig.add_subplot(gs[1, 1])

# Plot the curves and data on the main axes
ax_main.plot(
    norepi_log, result_variable_slope.best_fit,
    color='darkorange', lw=2, label='Variable Hill slope')
ax_main.plot(
    norepi_log, result_fixed_slope.best_fit,
    color='forestgreen', lw=2, label='Fixed Hill slope')
ax_main.plot(
    norepi_log, relaxation, 'ko', label='Data')
ax_main.set_title("Comparing model fits")
ax_main.set_xlabel("Log[Norepinephrine, M]")
ax_main.set_ylabel(r"Relaxation (%)")
ax_main.legend()

# Plot the residuals on their respective axes
result_variable_slope.plot_residuals(
    ax=ax_resid_var, data_kws={'color': 'darkorange'})
ax_resid_var.set_ylabel("Residuals")
ax_resid_var.set_ylim((-12, 10))
ax_resid_var.set_title("Variable Hill slope")

result_fixed_slope.plot_residuals(
    ax=ax_resid_fix, data_kws={'color': 'forestgreen'})
ax_resid_fix.set_ylabel("Residuals")
ax_resid_fix.set_ylim((-12, 10))
ax_resid_fix.set_title("Fixed Hill slope")

plt.tight_layout();

#### Analysis of variance


In [ ]:
print("Sums of squares:")
print(f"Model\t\tχ²\tDF")
print(f"Reduced 4PL\t{result_fixed_slope.chisqr:.1f}\
\t{result_fixed_slope.nfree}")
print(f"Reduced 3PL\t{result_variable_slope.chisqr:.1f}\
\t{result_variable_slope.nfree}")

# Calculate sum of squares and DF differences
ss_difference = result_fixed_slope.chisqr - result_variable_slope.chisqr
df_difference = result_fixed_slope.nfree - result_variable_slope.nfree

print(f"Difference\t{ss_difference:.1f}\t{df_difference}")

#### Mean squares


In [ ]:
# Mean square for the fixed slope model (MSE_fixed)
ms_fixed = result_fixed_slope.chisqr / result_fixed_slope.nfree

# Mean square for the variable slope model (MSE_variable)
ms_variable = result_variable_slope.chisqr / result_variable_slope.nfree

# Mean square for the difference between the models (MSR)
ms_difference = ss_difference/ df_difference

print("Mean squares:")
print(f" MS (fixed Hill slope) = {ms_fixed:.2f}")
print(f" MS (variable Hill slope) = {ms_variable:.2f}")
print(f" MS (difference) = {ms_difference:.2f}")

#### F-ratio


In [ ]:
# Calculate the F-ratio
f_value = ms_difference / ms_variable
print(f"F ratio = {f_value:.3f}")

#### Calculating the P value


In [ ]:
from scipy.stats import f as f_dist

# Calculate the P value using the survival function of the F-distribution
p_value = f_dist(
    dfn=df_difference, dfd=result_variable_slope.nfree).sf(f_value)

print(f"P value = {p_value:.7f}")

#### Visualizing the F-distribution for P value and critical value


In [ ]:

# Calculate critical F-value
f_crit = f_dist(
    dfn=df_difference, dfd=result_variable_slope.nfree).ppf(1 - α)

# Generate x values for plotting
x_f = np.linspace(0, 40, 500)
hx_f = f_dist.pdf(x_f, df_difference, result_variable_slope.nfree)

plt.figure(figsize=(3.5, 3))

# Create the plot
plt.plot(x_f, hx_f, lw=2, color='black')

# Critical value
plt.axvline(
    x=f_crit,
    color='orangered', linestyle='--',
    label=f"F* ({f_crit:.2f})")

# Alpha area
plt.fill_between(
    x_f[x_f >= f_crit],
    hx_f[x_f >= f_crit],
    color='tomato',
    label=f"α ({α})")

# F-statistic
plt.axvline(
    x=f_value,
    color='limegreen', linestyle='-.',
    label=f"F ({f_value:.2f})")

# P-value area
plt.fill_between(
    x_f[x_f >= f_value],
    hx_f[x_f >= f_value],
    color='greenyellow',
    label=f"P ({p_value:.3f})")

plt.xlabel("F")
plt.ylabel('Density')
plt.ylim(0, .1)
plt.title(f"F-distribution (DFn={df_difference}, \
DFd={result_variable_slope.nfree})")
plt.margins(x=0.05, y=0)
plt.legend();

#### ANOVA table


### Model selection criteria


In [ ]:
# Print the AIC and BIC values for the variable slope model
print("Variable Hill slope (reduced 4PL) model:")
print(f"AIC: {result_variable_slope.aic:.2f}")
print(f"BIC: {result_variable_slope.bic:.2f}")
print()

# Print the AIC and BIC values for the fixed slope model
print("Fixed Hill slope (reduced 3PL) model:")
print(f"AIC: {result_fixed_slope.aic:.2f}")
print(f"BIC: {result_fixed_slope.bic:.2f}")

### Overfitting


## Bootstrapping for nonlinear regression
### Specificity of the nonlinear regression


### Generating bootstrap samples


In [ ]:
def draw_bs_pairs_nonlinreg(x, y, model_function, initial_params, size=1):
    """
    Perform pairs bootstrap for nonlinear regression and return parameters
    """

    inds = np.arange(len(x))

    # Array to store bootstrap parameter estimates
    bs_params_reps = np.empty((size, len(initial_params)))

    for i in range(size):
        bs_inds = np.random.choice(inds, len(inds), replace=True)
        bs_x, bs_y = x[bs_inds], y[bs_inds]

        # Fit the nonlinear regression model with SciPy `curve_fit`
        try:
            best_vals, _ = curve_fit(
            model_function, bs_x, bs_y, p0=initial_params)
        except RuntimeError as e:
            # Handle cases where the fit might not converge
            print(
                f"Warning: fit did not converge for bs sample {i+1}:\n {e}")

        bs_params_reps[i, :] = best_vals

    return bs_params_reps

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Set the number of bootstrap replicates
B = 10_000

# Generate bootstrap replicates of the parameters
bs_params_reps = draw_bs_pairs_nonlinreg(
    x=norepi_log,
    y=relaxation,
    model_function=hill_equation_bottom_zero,
    initial_params=initial_guesses,
    size=B)

# Print the first 5 replicates for each parameter
print("\nParameters from the first bootstrap replicates:")
print(" Top: ", bs_params_reps[:5, 0].round(1))
print(" LogEC50: ", bs_params_reps[:5, 1].round(2))
print(" Hill slope: ", bs_params_reps[:5, 2].round(2))

In [ ]:
# Print the maximal value of 'top' parameter in the bootstrap replications
print(f"Maximal value of the top parameter in the bootstrap replicates: \
{np.max(bs_params_reps[:, 0]):.1f}")
# Use np.nanmax to calculate array max ignoring NaN values

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Set the number of bootstrap replicates
B = 10_000

# Generate bootstrap replicates of the parameters
bs_params_reps_filtered = draw_bs_pairs_nonlinreg(
    x=norepi_log,
    y=relaxation,
    model_function=hill_equation_bottom_zero,
    initial_params=initial_guesses,
    size=B)

# Replace unrealistic 'top' and 'hill_slope' values with NaN
for i, params in enumerate(bs_params_reps_filtered):
    if params[0] > 250 or np.abs(params[2]) > 10:
        print(f"Warning: unrealistic parameter values for bs sample {i+1}")
        bs_params_reps_filtered[i, :] = np.nan
        # Assign NaN values for the entire row

# Print the number of NaN bootstrap replicates
print(f"Number of invalid bootstrap replicates: \
{np.sum(np.isnan(bs_params_reps_filtered))}")

### Constructing percentile intervals


In [ ]:
# Calculate the means, SEs, and 95% percentile intervals for each parameter
# using the nan counterpart of mean, std and percentile, to ignore NaN
bs_params_means = np.nanmean(bs_params_reps_filtered, axis=0)
bs_params_se = np.nanstd(bs_params_reps_filtered, axis=0, ddof=1)
bs_params_ci = np.nanpercentile(
    bs_params_reps_filtered, [2.5, 97.5], axis=0)

# Print the results
print("Bootstrap statistics:")
for i, param_name in enumerate(['top', 'LogEC50', 'Hill slope']):
    print(f"{param_name}:")
    print(f" mean = {bs_params_means[i].round(1)}")
    print(f" SE = {bs_params_se[i].round(4)}")
    print(f" 95% percentile interval = {bs_params_ci[:, i].round(3)}")

In [ ]:

# Create a figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(8.5, 3))

# Plot the bootstrap distributions for each parameter
for i, param_name in enumerate(['top', 'logEC50', 'hill_slope']):
    axes[i].hist(
        bs_params_reps[:, i],
        bins='auto', density=False,
        color='gold',
        label=f'Bootstrap replicates')
    axes[i].axvline(
        x=result.params[param_name].value,
        color='cornflowerblue', linestyle='-', lw=2,
        label=f'Observed ({result.params[param_name].value:.2f})')
    axes[i].axvline(
        x=bs_params_ci[0, i],
        color='red',
        linestyle='--',
        label=f"2.5th ({bs_params_ci[0, i]:.1f})")
    axes[i].axvline(
        x=bs_params_ci[1, i],
        color='red',
        linestyle='-.',
        label=f"97.5th ({bs_params_ci[1, i]:.1f})")
    # Adjust slightly the x-axis limits
    if param_name == 'top':
        axes[i].set_xlim((100, 115))
    elif param_name == 'logEC50':
        axes[i].set_xlim((-5.8, -5.4))
    else: axes[i].set_xlim((.45, .75))

    axes[i].set_xlabel(param_name)
    axes[i].set_ylabel('Frequency')
    axes[i].set_title(f"Bootstrap {param_name}")
    axes[i].legend(fontsize=8)

plt.tight_layout();

### Confidence band visualization


In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the variability of the fitted curves
for i in range(200):
    # Generate predicted values using bootstrap parameter replicates
    predicted_values = hill_equation_bottom_zero(
        x_range, *bs_params_reps_filtered[i, :])
    plt.plot(
        10**x_range, predicted_values,
        color='royalblue', lw=1, alpha=.4,
        # Print label only once
        label='Bootstrapped curves' if i==0 else None)

# Plot the original data and the best-fit curve
plt.plot(norepi_lin, relaxation, 'ko', label='Data')
plt.plot(
    10**x_range, hill_equation_bottom_zero(x_range, *best_vals),
    color="crimson", lw=2,
    label='Best fit')

plt.xscale('log')
plt.xlabel("[Norepinephrine, M]")
plt.ylabel("% relaxation")
plt.title("Bootstrap confidence band")
plt.legend();

## Conclusion
